# LiDARRGBPointCloudGenerator 使用演示

本 notebook 展示 LiDARRGBPointCloudGenerator 的基本使用方式和核心功能，特别是静动态分割功能。

## 功能概述

### LiDARRGBPointCloudGenerator
1. 从 MultiSceneDataset 的段中加载 LiDAR 点云
2. 通过多相机投影获取 RGB 颜色
3. **静动态分割**：将点云分割为静态背景和动态物体
4. 静态点保存为世界坐标，动态点保存为物体局部坐标
5. 支持多帧融合和动态物体优先渲染

## 核心特性

- **静态背景点**：不属于任何实例边界框的点，保存为世界坐标
- **动态物体点**：属于某个实例边界框的点，保存为物体局部坐标
- **多相机融合**：通过多相机投影提高 RGB 着色覆盖率
- **实例ID映射**：原始实例ID映射到连续int ID（从1开始）

## 使用说明

1. 按顺序执行所有单元格
2. 在"数据配置"单元格中修改配置文件路径
3. 每个部分可以独立运行和调试
4. 注意内存使用，特别是点云生成部分


## 第一部分：环境配置和数据准备

### 1. 环境配置

安装和导入所有必要的依赖包。


In [1]:
# 安装依赖（如果需要）
# !pip install numpy matplotlib open3d omegaconf torch

import os
import sys
import numpy as np
import torch
from omegaconf import OmegaConf
from typing import List, Dict, Optional, Tuple
import matplotlib.pyplot as plt
import open3d as o3d
from tqdm import tqdm

# 添加项目路径以导入模块
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)

# 导入项目模块
from datasets.multi_scene_dataset import MultiSceneDataset
from datasets.pointcloud_generators.rgb_pointcloud_generator import LiDARRGBPointCloudGenerator

# 设置设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 设置随机种子（可选，用于可重复性）
torch.manual_seed(42)
np.random.seed(42)

print("Environment setup completed!")


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Using device: cuda
Environment setup completed!


### 2. 数据配置准备

读取配置文件，准备数据配置。


In [2]:
# 读取配置文件
config_path = os.path.join(project_root, "configs/evolsplat/multi_scene.yaml")
cfg = OmegaConf.load(config_path)

# 提取数据配置
data_cfg = cfg.data

# 提取 MultiSceneDataset 配置
multi_scene_cfg = cfg.multi_scene

# 显示配置信息
print("Data configuration:")
print(f"  Data root: {data_cfg.data_root}")
print(f"  Dataset type: {data_cfg.dataset}")
print(f"  Train scene IDs: {data_cfg.train_scene_ids}")
print(f"  Eval scene IDs: {data_cfg.eval_scene_ids}")

print("\nMultiSceneDataset configuration:")
print(f"  Num source keyframes: {multi_scene_cfg.num_source_keyframes}")
print(f"  Num target keyframes: {multi_scene_cfg.num_target_keyframes}")
print(f"  Segment overlap ratio: {multi_scene_cfg.segment_overlap_ratio}")
print(f"  Min keyframes per scene: {multi_scene_cfg.min_keyframes_per_scene}")
print(f"  Min keyframes per segment: {multi_scene_cfg.min_keyframes_per_segment}")
print(f"  Fixed segment AABB: {multi_scene_cfg.fixed_segment_aabb}")

# 准备 fixed_segment_aabb（如果配置了）
fixed_segment_aabb = None
if multi_scene_cfg.fixed_segment_aabb is not None:
    fixed_segment_aabb = torch.tensor(multi_scene_cfg.fixed_segment_aabb, dtype=torch.float32)
    print(f"\nUsing fixed segment AABB: {fixed_segment_aabb}")

# 准备点云生成器的边界框配置（从 pointcloud 配置中获取，如果存在）
crop_aabb = None
input_aabb = None
if hasattr(data_cfg, 'pointcloud'):
    if 'crop_aabb' in data_cfg.pointcloud:
        crop_aabb = np.array(data_cfg.pointcloud.crop_aabb)
    if 'input_aabb' in data_cfg.pointcloud:
        input_aabb = np.array(data_cfg.pointcloud.input_aabb)


Data configuration:
  Data root: /mnt/f/DataSet/nuScenes/processed/trainval
  Dataset type: nuscenes
  Train scene IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
  Eval scene IDs: [10, 11, 12, 13, 14]

MultiSceneDataset configuration:
  Num source keyframes: 3
  Num target keyframes: 6
  Segment overlap ratio: 0.2
  Min keyframes per scene: 10
  Min keyframes per segment: 6
  Fixed segment AABB: [[-20, -20, -5], [70, 20, 5]]

Using fixed segment AABB: tensor([[-20., -20.,  -5.],
        [ 70.,  20.,   5.]])


### 3. 创建 MultiSceneDataset 实例

创建 MultiSceneDataset 实例，用于加载数据。


In [3]:
# 创建 MultiSceneDataset 实例
dataset = MultiSceneDataset(
    data_cfg=data_cfg,
    train_scene_ids=data_cfg.train_scene_ids,
    eval_scene_ids=data_cfg.eval_scene_ids,
    num_source_keyframes=multi_scene_cfg.num_source_keyframes,
    num_target_keyframes=multi_scene_cfg.num_target_keyframes,
    segment_overlap_ratio=multi_scene_cfg.segment_overlap_ratio,
    keyframe_split_config=dict(multi_scene_cfg.keyframe_split_config),
    min_keyframes_per_scene=multi_scene_cfg.min_keyframes_per_scene,
    min_keyframes_per_segment=multi_scene_cfg.min_keyframes_per_segment,
    device=device,
    preload_scene_count=1,  # 预加载1个场景
    fixed_segment_aabb=fixed_segment_aabb,
)

print("MultiSceneDataset created successfully!")

# 初始化数据集
dataset.initialize()

# 获取当前场景ID
current_scene_id = dataset.get_current_scene_id()
print(f"Current training scene ID: {current_scene_id}")

if current_scene_id is not None:
    print(f"Dataset initialized successfully with scene {current_scene_id}")
else:
    print("Warning: No valid training scenes found after validation")


MultiSceneDataset created successfully!


Loading lidar: 100%|██████████| 196/196 [00:02<00:00, 92.50it/s]
Projecting lidar pts on images for camera CAM_FRONT: 100%|██████████| 196/196 [00:01<00:00, 119.54it/s]
Projecting lidar pts on images for camera CAM_FRONT_LEFT: 100%|██████████| 196/196 [00:01<00:00, 112.60it/s]
Projecting lidar pts on images for camera CAM_FRONT_RIGHT: 100%|██████████| 196/196 [00:01<00:00, 115.78it/s]
Loading lidar: 100%|██████████| 196/196 [00:03<00:00, 61.72it/s]
Projecting lidar pts on images for camera CAM_FRONT: 100%|██████████| 196/196 [00:02<00:00, 90.56it/s]
Projecting lidar pts on images for camera CAM_FRONT_LEFT: 100%|██████████| 196/196 [00:02<00:00, 91.31it/s]
Projecting lidar pts on images for camera CAM_FRONT_RIGHT: 100%|██████████| 196/196 [00:02<00:00, 93.29it/s]
Loading lidar: 100%|██████████| 196/196 [00:02<00:00, 95.55it/s]
Projecting lidar pts on images for camera CAM_FRONT: 100%|██████████| 196/196 [00:02<00:00, 93.14it/s]
Projecting lidar pts on images for camera CAM_FRONT_LEFT: 1

Current training scene ID: 8
Dataset initialized successfully with scene 8


## 第二部分：LiDARRGBPointCloudGenerator 基本使用

### 1. 创建 LiDARRGBPointCloudGenerator 实例

创建点云生成器，配置参数说明：
- `chosen_cam_ids`: 选择使用的相机ID列表（默认根据数据集类型设置）
- `camera_priority`: 相机优先级（用于RGB着色，优先级高的覆盖优先级低的）
- `resomult`: 图像分辨率缩放倍数（默认 0.5，用于加速投影计算）
- `dataset`: 数据集类型（waymo/kitti/nuscenes/argoverse）
- `crop_aabb`: 裁剪边界框，用于裁剪时移除超出边界框的点云
- `input_aabb`: 输入边界框，用于分割和滤波时区分内部和外部点云
- `use_bbx`: 是否使用边界框裁剪


In [4]:
# 创建 LiDARRGBPointCloudGenerator 实例
# 如果没有配置 crop_aabb 和 input_aabb，使用默认值
if crop_aabb is None:
    # 默认裁剪边界框（根据数据集类型调整）
    crop_aabb = np.array([[-20, -20, -20], [20, 20, 70]], dtype=np.float32)
if input_aabb is None:
    # 默认输入边界框
    input_aabb = np.array([[-20, -20, -20], [20, 20, 70]], dtype=np.float32)

pointcloud_generator = LiDARRGBPointCloudGenerator(
    chosen_cam_ids=None,  # 使用默认值（根据数据集类型设置）
    camera_priority=None,  # 使用默认优先级
    resomult=0.5,  # 图像分辨率缩放倍数
    dataset=data_cfg.dataset,  # 数据集类型
    crop_aabb=crop_aabb,
    input_aabb=input_aabb,
    use_bbx=True,  # 使用边界框裁剪
    device=device,
)

print("LiDARRGBPointCloudGenerator created successfully!")
print(f"  Dataset type: {pointcloud_generator.dataset}")
print(f"  Chosen camera IDs: {pointcloud_generator.chosen_cam_ids}")
print(f"  Camera priority: {pointcloud_generator.camera_priority}")
print(f"  Resolution multiplier: {pointcloud_generator.resomult}")


LiDARRGBPointCloudGenerator created successfully!
  Dataset type: nuscenes
  Chosen camera IDs: [0, 1, 2, 3, 4, 5]
  Camera priority: [0, 1, 2, 3, 4, 5]
  Resolution multiplier: 0.5


### 2. 获取场景和段信息

选择一个场景和段用于生成点云。


In [5]:
# 选择一个场景和段
scene_id = data_cfg.train_scene_ids[0] if len(data_cfg.train_scene_ids) > 0 else None
segment_id = 0  # 使用第一个段

if scene_id is not None:
    scene_info = dataset.get_scene(scene_id)
    
    if scene_info is not None:
        print(f"Scene {scene_id} information:")
        print(f"  Number of frames: {scene_info['num_frames']}")
        print(f"  Number of cameras: {scene_info['num_cams']}")
        print(f"  Number of segments: {len(scene_info['segments'])}")
        
        # 显示段信息
        if segment_id < len(scene_info['segments']):
            segment = scene_info['segments'][segment_id]
            print(f"\nSegment {segment_id} information:")
            print(f"  Number of frames: {len(segment['frame_indices'])}")
            print(f"  Frame indices (first 10): {segment['frame_indices'][:10]}")
            print(f"  AABB min: {segment['aabb'][0]}")
            print(f"  AABB max: {segment['aabb'][1]}")
        else:
            print(f"\nWarning: Segment {segment_id} not found, using segment 0")
            segment_id = 0
    else:
        print(f"Scene {scene_id} not found")
        scene_id = None
else:
    print("No training scenes available")


Loading lidar: 100%|██████████| 196/196 [00:03<00:00, 62.99it/s]
Projecting lidar pts on images for camera CAM_FRONT: 100%|██████████| 196/196 [00:02<00:00, 95.24it/s]
Projecting lidar pts on images for camera CAM_FRONT_LEFT: 100%|██████████| 196/196 [00:02<00:00, 86.01it/s]
Projecting lidar pts on images for camera CAM_FRONT_RIGHT: 100%|██████████| 196/196 [00:00<00:00, 368.84it/s]


Scene 0 information:
  Number of frames: 196
  Number of cameras: 3
  Number of segments: 2

Segment 0 information:
  Number of frames: 105
  Frame indices (first 10): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
  AABB min: tensor([-20., -20.,  -5.], device='cuda:0')
  AABB max: tensor([70., 20.,  5.], device='cuda:0')


### 3. 生成点云（包含静动态分割）

使用 `generate_pointcloud_with_static_dynamic()` 方法生成点云，该方法返回：
- `frame_points`: List[np.ndarray] - 每项为 (N, 6) 世界坐标背景点 + RGB
- `waymoid2intid`: Dict[int, int] - 原始实例ID -> 连续int ID（从1开始）
- `intid2inboxpoints`: Dict[int, Dict[int, np.ndarray]] - 动态物体点字典
  - `intid2inboxpoints[intid][frame_idx] = (N, 6)` 局部坐标 + RGB


In [6]:
# 生成点云（包含静动态分割）
if scene_id is not None:
    print(f"Generating point cloud for Scene {scene_id}, Segment {segment_id}...")
    print("This may take a while depending on the number of frames...")
    
    frame_points, waymoid2intid, intid2inboxpoints = pointcloud_generator.generate_pointcloud_with_static_dynamic(
        dataset=dataset,
        scene_id=scene_id,
        segment_id=segment_id,
    )
    
    print("\nPoint cloud generation completed!")
    print(f"\nResults summary:")
    print(f"  Number of frames: {len(frame_points)}")
    
    # 统计静态点
    total_static_points = sum(len(fp) for fp in frame_points)
    print(f"  Total static points: {total_static_points:,}")
    if len(frame_points) > 0:
        avg_static_points = total_static_points / len(frame_points)
        print(f"  Average static points per frame: {avg_static_points:,.0f}")
    
    # 统计动态点
    total_dynamic_instances = len(intid2inboxpoints)
    print(f"  Number of dynamic instances: {total_dynamic_instances}")
    
    total_dynamic_points = 0
    for intid, frame_dict in intid2inboxpoints.items():
        for frame_idx, points in frame_dict.items():
            total_dynamic_points += len(points)
    print(f"  Total dynamic points: {total_dynamic_points:,}")
    
    # 显示实例ID映射
    if len(waymoid2intid) > 0:
        print(f"\nInstance ID mapping (showing first 10):")
        for i, (waymoid, intid) in enumerate(list(waymoid2intid.items())[:10]):
            print(f"    Waymo ID {waymoid} -> Internal ID {intid}")
        if len(waymoid2intid) > 10:
            print(f"    ... and {len(waymoid2intid) - 10} more")
    
    # 显示动态实例信息
    if total_dynamic_instances > 0:
        print(f"\nDynamic instances information (showing first 5):")
        for i, (intid, frame_dict) in enumerate(list(intid2inboxpoints.items())[:5]):
            total_points_for_instance = sum(len(points) for points in frame_dict.values())
            print(f"    Instance {intid}: {len(frame_dict)} frames, {total_points_for_instance:,} total points")
        if total_dynamic_instances > 5:
            print(f"    ... and {total_dynamic_instances - 5} more instances")
else:
    print("Cannot generate point cloud: scene not available")
    frame_points = []
    waymoid2intid = {}
    intid2inboxpoints = {}


Generating point cloud for Scene 0, Segment 0...
This may take a while depending on the number of frames...

Point cloud generation completed!

Results summary:
  Number of frames: 105
  Total static points: 971,867
  Average static points per frame: 9,256
  Number of dynamic instances: 0
  Total dynamic points: 0

Instance ID mapping (showing first 10):
    Waymo ID 0 -> Internal ID 1
    Waymo ID 1 -> Internal ID 2
    Waymo ID 2 -> Internal ID 3
    Waymo ID 3 -> Internal ID 4
    Waymo ID 4 -> Internal ID 5
    Waymo ID 5 -> Internal ID 6
    Waymo ID 6 -> Internal ID 7
    Waymo ID 7 -> Internal ID 8
    Waymo ID 8 -> Internal ID 9
    Waymo ID 9 -> Internal ID 10
    ... and 9 more


## 第三部分：可视化点云

### 1. 可视化静态背景点云

合并所有帧的静态点，创建 Open3D 点云对象并可视化。


In [7]:
# 合并所有帧的静态点
if len(frame_points) > 0:
    print("Merging static background points from all frames...")
    static_points = np.concatenate(frame_points, axis=0)
    print(f"Total static points: {static_points.shape[0]:,}")
    
    # 创建 Open3D 点云对象
    static_pcd = o3d.geometry.PointCloud()
    static_pcd.points = o3d.utility.Vector3dVector(static_points[:, :3])
    
    # 确保颜色在 [0, 1] 范围内
    colors = static_points[:, 3:6].astype(np.float32)
    if colors.max() > 1.0:
        colors = colors / 255.0
    colors = np.clip(colors, 0.0, 1.0)
    static_pcd.colors = o3d.utility.Vector3dVector(colors)
    
    print("Static point cloud created successfully!")
    print(f"  Number of points: {len(static_pcd.points)}")
    print(f"  Has colors: {static_pcd.has_colors()}")
    
    # 显示点云边界
    if len(static_pcd.points) > 0:
        bbox = static_pcd.get_axis_aligned_bounding_box()
        print(f"  Bounding box min: {bbox.min_bound}")
        print(f"  Bounding box max: {bbox.max_bound}")
else:
    print("No static points to visualize")
    static_pcd = None


Merging static background points from all frames...
Total static points: 971,867
Static point cloud created successfully!
  Number of points: 971867
  Has colors: True
  Bounding box min: [-130.16876221  -14.49935913  -44.85289764]
  Bounding box max: [104.5897522    3.3442533  106.72044373]


### 2. 可视化静态点云（使用 Open3D）

使用 Open3D 的可视化功能显示静态背景点云。


In [8]:
# 可视化点云
if static_pcd is not None and len(static_pcd.points) > 0:
    print("Visualizing static point cloud...")
    
    # 方法1: 尝试使用 draw_geometries（自动检测环境，支持Jupyter）
    try:
        # 在Jupyter环境中，draw_geometries 应该使用WebVisualizer
        o3d.visualization.draw_geometries(
            [static_pcd],
            window_name=f"Static Point Cloud - Scene {scene_id}, Segment {segment_id}",
            width=800,
            height=600,
            point_show_normal=False,
        )
        print("Visualization completed using draw_geometries!")
    except Exception as e1:
        print(f"draw_geometries failed: {e1}")
        print("Trying alternative visualization method...")
        
        # 方法2: 尝试使用Visualizer（适用于有GUI的环境）
        vis = None
        try:
            vis = o3d.visualization.Visualizer()
            # 尝试创建窗口，如果失败会抛出异常
            if not vis.create_window(
                window_name=f"Static Point Cloud - Scene {scene_id}, Segment {segment_id}",
                width=800,
                height=600,
                visible=True
            ):
                raise RuntimeError("Failed to create visualization window")
            
            vis.add_geometry(static_pcd)
            
            # 设置视角
            view_ctl = vis.get_view_control()
            view_ctl.set_front([0, 0, -1])
            view_ctl.set_lookat([0, 0, 0])
            view_ctl.set_up([0, -1, 0])
            view_ctl.set_zoom(0.7)
            
            # 渲染（阻塞直到窗口关闭）
            vis.run()
            print("Visualization completed using Visualizer!")
            
        except Exception as e2:
            print(f"Visualizer also failed: {e2}")
            print("\n" + "="*60)
            print("GUI visualization is not available in this environment.")
            print("Alternative options:")
            print("="*60)
            
            # 方法3: 使用matplotlib进行简单的3D可视化
            try:
                from mpl_toolkits.mplot3d import Axes3D
                
                points = np.asarray(static_pcd.points)
                colors = np.asarray(static_pcd.colors) if static_pcd.has_colors() else None
                
                # 下采样以加快可视化（如果点太多）
                max_points = 10000
                if len(points) > max_points:
                    indices = np.random.choice(len(points), max_points, replace=False)
                    points = points[indices]
                    if colors is not None:
                        colors = colors[indices]
                    print(f"Downsampled to {max_points} points for visualization")
                
                fig = plt.figure(figsize=(12, 10))
                ax = fig.add_subplot(111, projection='3d')
                
                if colors is not None:
                    ax.scatter(points[:, 0], points[:, 1], points[:, 2], 
                              c=colors, s=1, alpha=0.6)
                else:
                    ax.scatter(points[:, 0], points[:, 1], points[:, 2], 
                              s=1, alpha=0.6)
                
                ax.set_xlabel('X')
                ax.set_ylabel('Y')
                ax.set_zlabel('Z')
                ax.set_title(f'Static Point Cloud - Scene {scene_id}, Segment {segment_id}')
                
                plt.tight_layout()
                plt.show()
                print("Visualization completed using matplotlib!")
                
            except Exception as e3:
                print(f"Matplotlib visualization also failed: {e3}")
                print("\nPlease use an external tool to visualize the point cloud.")
                print(f"The point cloud has been saved (if save cell was executed).")
                print(f"You can use tools like CloudCompare, MeshLab, or Open3D viewer to open the PLY file.")
        finally:
            # 确保窗口正确关闭
            if vis is not None:
                try:
                    vis.destroy_window()
                except:
                    pass
                try:
                    del vis
                except:
                    pass
else:
    print("No static point cloud to visualize")


Visualizing static point cloud...
Visualization completed using draw_geometries!


In [ ]:
# 可视化动态物体点云
# 注意：动态点存储在局部坐标系中，需要转换到世界坐标系才能可视化
# 这里我们展示如何访问动态点数据，实际转换需要获取实例的位姿信息

if len(intid2inboxpoints) > 0:
    print("Dynamic objects information:")
    print(f"  Number of dynamic instances: {len(intid2inboxpoints)}")
    
    # 显示每个实例的点云信息
    for intid, frame_dict in list(intid2inboxpoints.items())[:5]:  # 只显示前5个实例
        total_points = sum(len(points) for points in frame_dict.values())
        print(f"\n  Instance {intid}:")
        print(f"    Frames with points: {list(frame_dict.keys())}")
        print(f"    Total points: {total_points:,}")
        
        # 显示第一帧的点云示例
        if len(frame_dict) > 0:
            first_frame_idx = list(frame_dict.keys())[0]
            first_frame_points = frame_dict[first_frame_idx]
            print(f"    First frame ({first_frame_idx}) points shape: {first_frame_points.shape}")
            print(f"    First frame points range (local coords):")
            print(f"      X: [{first_frame_points[:, 0].min():.2f}, {first_frame_points[:, 0].max():.2f}]")
            print(f"      Y: [{first_frame_points[:, 1].min():.2f}, {first_frame_points[:, 1].max():.2f}]")
            print(f"      Z: [{first_frame_points[:, 2].min():.2f}, {first_frame_points[:, 2].max():.2f}]")
    
    print("\nNote: Dynamic points are stored in local (object) coordinates.")
    print("To visualize them in world coordinates, you need to:")
    print("  1. Get the instance pose (T_ow) from the dataset")
    print("  2. Transform local points to world: p_world = T_ow @ p_local")
    print("  3. Then visualize the transformed points")
else:
    print("No dynamic objects found in this segment")


### 4. 保存点云到文件

将生成的点云保存为 PLY 文件，方便后续使用外部工具查看。


In [ ]:
# 保存静态点云
if static_pcd is not None and len(static_pcd.points) > 0:
    output_dir = os.path.join(project_root, "outputs", "pointclouds")
    os.makedirs(output_dir, exist_ok=True)
    
    static_output_path = os.path.join(
        output_dir, 
        f"scene_{scene_id}_segment_{segment_id}_static.ply"
    )
    
    success = o3d.io.write_point_cloud(static_output_path, static_pcd)
    if success:
        print(f"Static point cloud saved to: {static_output_path}")
    else:
        print(f"Failed to save static point cloud to: {static_output_path}")
else:
    print("No static point cloud to save")


## 第四部分：高级功能演示

### 1. 使用基类接口生成点云

`generate_pointcloud()` 方法返回合并后的静态点云（符合基类接口），不包含动态物体点。


In [ ]:
# 使用基类接口生成点云（只返回静态点云）
if scene_id is not None:
    print("Generating point cloud using base class interface...")
    pointcloud = pointcloud_generator.generate_pointcloud(
        dataset=dataset,
        scene_id=scene_id,
        segment_id=segment_id,
    )
    
    print("Point cloud generation completed!")
    print(f"  Number of points: {len(pointcloud.points)}")
    print(f"  Has colors: {pointcloud.has_colors()}")
    
    if len(pointcloud.points) > 0:
        bbox = pointcloud.get_axis_aligned_bounding_box()
        print(f"  Bounding box min: {bbox.min_bound}")
        print(f"  Bounding box max: {bbox.max_bound}")
        
        # 保存点云
        output_path = os.path.join(
            project_root, "outputs", "pointclouds",
            f"scene_{scene_id}_segment_{segment_id}_base_interface.ply"
        )
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        success = o3d.io.write_point_cloud(output_path, pointcloud)
        if success:
            print(f"  Point cloud saved to: {output_path}")
else:
    print("Cannot generate point cloud: scene not available")


### 2. 按帧查看静态点云

查看每一帧的静态点云，了解点云随时间的分布。


In [ ]:
# 按帧查看静态点云
if len(frame_points) > 0:
    print("Frame-by-frame static point cloud statistics:")
    print(f"{'Frame':<8} {'Points':<12} {'X Range':<25} {'Y Range':<25} {'Z Range':<25}")
    print("-" * 100)
    
    for i, fp in enumerate(frame_points[:10]):  # 只显示前10帧
        if len(fp) > 0:
            points = fp[:, :3]
            x_range = f"[{points[:, 0].min():.2f}, {points[:, 0].max():.2f}]"
            y_range = f"[{points[:, 1].min():.2f}, {points[:, 1].max():.2f}]"
            z_range = f"[{points[:, 2].min():.2f}, {points[:, 2].max():.2f}]"
            print(f"{i:<8} {len(fp):<12} {x_range:<25} {y_range:<25} {z_range:<25}")
        else:
            print(f"{i:<8} {0:<12} {'N/A':<25} {'N/A':<25} {'N/A':<25}")
    
    if len(frame_points) > 10:
        print(f"... and {len(frame_points) - 10} more frames")
else:
    print("No frame points to display")


### 3. 查看动态物体在不同帧中的分布

查看每个动态物体在不同帧中的点云分布情况。


In [ ]:
# 查看动态物体在不同帧中的分布
if len(intid2inboxpoints) > 0:
    print("Dynamic objects frame distribution:")
    print(f"{'Instance ID':<15} {'Frames':<30} {'Total Points':<15} {'Avg Points/Frame':<20}")
    print("-" * 80)
    
    for intid, frame_dict in list(intid2inboxpoints.items())[:10]:  # 只显示前10个实例
        frame_indices = sorted(frame_dict.keys())
        total_points = sum(len(points) for points in frame_dict.values())
        avg_points = total_points / len(frame_dict) if len(frame_dict) > 0 else 0
        
        frames_str = str(frame_indices[:5])  # 只显示前5个帧索引
        if len(frame_indices) > 5:
            frames_str = frames_str[:-1] + f", ... ({len(frame_indices)} total)]"
        
        print(f"{intid:<15} {frames_str:<30} {total_points:<15} {avg_points:<20.1f}")
    
    if len(intid2inboxpoints) > 10:
        print(f"... and {len(intid2inboxpoints) - 10} more instances")
else:
    print("No dynamic objects to display")


## 总结

本 notebook 演示了 LiDARRGBPointCloudGenerator 的基本使用方式，包括：

1. **环境配置和数据准备**：设置环境、加载配置、创建数据集
2. **点云生成**：使用 `generate_pointcloud_with_static_dynamic()` 生成包含静动态分割的点云
3. **可视化**：使用 Open3D 和 matplotlib 可视化点云
4. **数据保存**：将点云保存为 PLY 文件

### 关键要点

- **静态点**：保存在世界坐标系中，可以跨帧累积
- **动态点**：保存在物体局部坐标系中，需要位姿信息才能转换到世界坐标
- **实例ID映射**：原始实例ID（waymoid）映射到连续int ID（从1开始）
- **多相机融合**：通过多相机投影提高 RGB 着色覆盖率

### 下一步

- 使用动态物体的位姿信息将局部坐标点转换到世界坐标
- 实现动态物体优先渲染（先渲染动态，再渲染静态）
- 使用 FrameSpec 选择特定帧进行渲染
- 集成到训练流程中使用
